<a href="https://colab.research.google.com/github/blssmx/CRM-Coursework-2/blob/main/CWK2_1g.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/pycaret/pycaret.git@master

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yfin

In [ ]:
#Task A Question A
ticker = 'AMZN'
ticker = yfin.Ticker(ticker)

data = ticker.history(period='5y')

In [ ]:
data

In [ ]:
plt.ylabel("Amazon")

data['Close'].plot(figsize=(10,5))

In [ ]:
#Task A Question B
data['SMA_20'] = data['Close'].rolling(window=20).mean()
data['SMA_50'] = data['Close'].rolling(window=50).mean()

data['EMA_20'] = data['Close'].ewm(span=20, adjust=False).mean()

delta = data['Close'].diff(1)
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
data['RSI'] = 100 - (100 / (1 + rs))

data

In [ ]:
data = data.dropna()



In [ ]:
data

In [ ]:
# Initialising X and assigning the two feature variables
X = data[['EMA_20','SMA_50']]

# Getting the head of the data
X.head()

In [ ]:
# Setting-up the dependent variable
y = data['Close']

# Getting the head of the data
y.head()

In [ ]:
#Task A Question C

# Setting the training set to 80% of the data
training = 0.7
t = int(training*len(data))

# Training dataset
X_train = X[:t]
y_train = y[:t]

# Testing dataset
X_test = X[t:]
y_test = y[t:]

In [ ]:
from pycaret.regression import *
s = setup(
    data=data[['EMA_20','SMA_50','Close']],
    target='Close',
    session_id=123,
    numeric_features=['EMA_20', 'SMA_50'],
    fold_strategy='timeseries', # Ensures temporal integrity
    data_split_shuffle=False,     # Prevents random shuffling
    n_jobs=-1
)


In [ ]:
models()

In [ ]:
rf = create_model('rf')

In [ ]:
tuned_rf = tune_model(rf)

In [ ]:
print(tuned_rf)

In [ ]:
plot_model(tuned_rf)

In [ ]:
plot_model(tuned_rf, plot = 'error')

In [ ]:
plot_model(tuned_rf, plot = 'feature')

In [ ]:
evaluate_model(tuned_rf)

In [ ]:
predict_model(tuned_rf)

In [ ]:
final_rf = finalize_model(tuned_rf)

In [ ]:
print(final_rf)

In [ ]:
predict_model(final_rf)

In [ ]:
unseenrf_predictions = predict_model(final_rf, data=data_test)
unseenrf_predictions.head()

In [ ]:
rf_results = unseenrf_predictions.copy()
rf_results['Actual_Close'] = y_test.values
rf_results.rename(columns={'prediction_label': 'Predicted_Close'}, inplace=True)

plt.figure(figsize=(10, 5))

plt.plot(rf_results.index, rf_results['Actual_Close'], label='Actual Price')
plt.plot(rf_results.index, rf_results['Predicted_Close'], label='Predicted Price')

plt.title("Random Forest – Actual vs Predicted Close Price")
plt.xlabel("Time")
plt.ylabel("Close Price")
plt.legend()
plt.grid(True)
plt.show()


